In [ ]:
import os
import subprocess

# Definizione del percorso del repository su Google Drive
REPO_DIR = "/content/drive/MyDrive/progettoLLM/CLARITY"

# ==========================================
# 0. SINCRONIZZAZIONE AUTOMATICA (SOLO AMBIENTE COLAB)
# ==========================================
try:
    import google.colab
    print("Ambiente Colab rilevato. Inizio procedura di sincronizzazione forzata con GitHub...")

    google.colab.drive.mount('/content/drive', force_remount=True)

    # Verifica dell'esistenza della directory di destinazione
    if os.path.exists(REPO_DIR):
        
        # 1. Acquisizione degli ultimi aggiornamenti dal repository remoto
        subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, capture_output=True, text=True)
        
        # 2. Ripristino forzato dello stato locale per farlo coincidere con il branch remoto
        # Nota: le modifiche locali non committate verranno eliminate.
        result = subprocess.run(
            ["git", "reset", "--hard", "origin/main"],
            cwd=REPO_DIR,
            capture_output=True,
            text=True
        )
        
        # 3. Pulizia di eventuali file non tracciati presenti nella working directory
        subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True, text=True)

        if result.returncode == 0:
            print(f"Sincronizzazione completata con successo. Output Git:\n{result.stdout.strip()}")
        else:
            print(f"Errore critico durante la sincronizzazione. Dettagli:\n{result.stderr.strip()}")

        # Impostazione della directory di lavoro attiva per le successive esecuzioni Python
        os.chdir(REPO_DIR)

    else:
        print(f"Errore: Il percorso {REPO_DIR} non è stato trovato. È necessario eseguire prima lo script di setup di base.")

except ImportError:
    print("Ambiente locale rilevato. Procedura di sincronizzazione automatica ignorata per preservare le modifiche locali.")

In [ ]:
import os
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = None

# ==========================================
# 1. SMART TOKEN LOADING
# ==========================================
try:
    from google.colab import userdata
    print("Colab environment detected. Fetching token from Secrets...")
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    print("Local environment detected. Reading token from .env file...")
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path) as f:
            for line in f:
                if line.startswith("HF_TOKEN="):
                    hf_token = line.split("=", 1)[1].strip().strip("'\"")
                    break
    else:
        print("No .env file found in the local directory.")

# ==========================================
# 2. AUTHENTICATION
# ==========================================
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print("Hugging Face login successful!")
else:
    print("Warning: HF_TOKEN not found! Model download will fail if it's gated.")

# ==========================================
# 3. CARICAMENTO DEL MODELLO (QUANTIZZAZIONE 4-BIT)
# ==========================================
model_id = "meta-llama/Llama-4-Scout-17B-16E" 
print(f"Configurazione della quantizzazione a 4-bit per {model_id}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Caricamento del modello {model_id} in corso...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.float16  # Aggiornato da torch_dtype per conformità con le nuove API
)
print("Caricamento completato con successo. Il modello è pronto in memoria.")